In [5]:
import jax
import jax.numpy as jnp
from function_utils import construct_func_from_intensity_matrix
from function_utils import construct_func_from_payment_matrix
from prototype_4 import semimarkov_solver as pt4_mks
from prototype_4 import compute_derivative
from prototype_4 import update_p, update_p_point

In [60]:
"""
@jax.jit
def mu_3(t, u):
    my_poly2 = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.01
    
    my_poly = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.09
    
    return my_poly + my_poly2
"""

@jax.jit
def mu_3(t, u):
    return 0.15 + 0 * u

In [61]:
intensity_matrix = [
    [None, mu_3, mu_3, mu_3, None],
    [None, None, mu_3, mu_3, mu_3],
    [None, None, None, None, None],
    [None, None, None, None, None],
    [None, None, None, None, None]
]


intensity = construct_func_from_intensity_matrix(intensity_matrix)
units = 1
discretization_unit = 12

In [93]:
solver_steps = discretization_unit * units
grid = jnp.linspace(
    0, units, solver_steps + 1, endpoint=True 
)
grid = jnp.expand_dims(grid, 0)
step_size = 1 / discretization_unit

outflow, inflow = intensity(0, grid)

p_point_0 = jnp.zeros_like(outflow[..., 0, 0, :-1])
p_0 = jnp.zeros_like(outflow[..., 0, :-1])
p_point_0 = p_point_0.at[..., 0].set(1)

In [94]:
t = jnp.expand_dims(jnp.array([0]), -1)
p, p_point = p_0, p_point_0

t_left = t

mu_plus = intensity(t_left, grid)
mu_minus = intensity(t_left, grid)

next_inflow, delta_p, delta_p_point = compute_derivative(p, p_point, mu_plus, mu_minus)

t += step_size

p_2 = update_p(p, delta_p, next_inflow, step_size)
p_point_2 = update_p_point(p_point, delta_p_point, step_size)

t_right = t

mu_plus = intensity(t_right, grid)
mu_minus = intensity(t_right, grid)

next_inflow_2, delta_p_2, delta_p_point_2 = compute_derivative(
    p_2, p_point_2, mu_plus, mu_minus
)

In [113]:
next_inflow_2 = 0.5 * (next_inflow + next_inflow_2 + delta_p_2[..., 1])
delta_p2 = 0.5 * (delta_p_2[..., 1:] + delta_p[..., :-1])
delta_p_point2 = 0.5 * (delta_p_point_2[..., 1:] + delta_p_point[..., :-1])

delta_p = delta_p.at[..., :-1].set(delta_p2)
delta_p_point = delta_p_point.at[..., :-1].set(delta_p_point2)

In [120]:
delta_p_2[0, 1]

Array([-0.005625, -0.      , -0.      , -0.      , -0.      , -0.      ,
       -0.      , -0.      , -0.      , -0.      , -0.      , -0.      ],      dtype=float32)

In [111]:
p_2[0, 1]

Array([0.0125, 0.    , 0.    , 0.    , 0.    , 0.    , 0.    , 0.    ,
       0.    , 0.    , 0.    , 0.    ], dtype=float32)

In [ ]:
0.45 * p_2[0, 1]

Array([0.005625, 0.      , 0.      , 0.      , 0.      , 0.      ,
       0.      , 0.      , 0.      , 0.      , 0.      , 0.      ],      dtype=float32)